# 01 · Hypothesis, trading setup v1, evaluation protocol, research ledger
Written before any feature, model or backtest was computed (Week 1 checkpoint).

In [1]:
import pandas as pd
import yaml
from etfmom.config import ROOT, TAB, setup
from etfmom.ledger import LEDGER, init_ledger

s = setup()
print(yaml.safe_dump(s["hypothesis"], allow_unicode=True, sort_keys=False))

entry_point: mechanism_first
mechanism: Slow-moving capital and investor under-reaction make 6-12 month relative
  performance across asset-class ETFs persist over the next month. The persistence
  is weakest right after volatility spikes, when crashed losers rebound sharply (momentum
  crashes), so a volatility state should gate it.
source_of_edge:
- flow_predictability
- risk_compensation
counterparty: Rebalancers and benchmark-constrained allocators who trade against trends
  on a calendar, and investors who anchor on stale prices.
falsification:
- Rule-baseline and deployed-model validation IC: Newey-West 95% CI includes 0  ->
    no signal, stop.
- Break-even cost below 2x the assumed per-leg cost                             ->
  not tradable, stop.
- Holdout net Sharpe < 0 or holdout mean IC <= 0                                 ->
  not confirmed.
- Momentum IC NOT lower in the high-volatility tercile than in calm terciles     ->
  gate mechanism rejected.



## Evaluation protocol (six design commitments + split plan, calendar dates)

In [2]:
ev = s["evaluation"]
rows = [
    ("Window", ev["window"] + f" from {ev['first_train_date']}"),
    ("Step size", ev["step"]),
    ("Retraining cadence", ev["retrain_cadence"]),
    ("Label horizon", f"{ev['label_horizon_sessions']} sessions (open t+1 -> open t+22)"),
    ("Label buffer (purge)", f"{ev['label_buffer_sessions']} sessions"),
    ("Feature buffer (embargo)", f"{ev['feature_embargo_sessions']} (training never follows validation)"),
    ("Longest feature lookback", f"{ev['longest_feature_lookback_sessions']} sessions"),
    ("Discovery (feature triage)", " to ".join(ev["discovery_period"])),
    ("HP tuning (fixed split)", f"train <= {ev['tuning_split']['train_end']}, valid {' to '.join(ev['tuning_split']['valid'])}"),
    ("Validation folds", f"{ev['validation_folds'][0]}-{ev['validation_folds'][-1]} (one calendar year each)"),
    ("Holdout (sealed)", " to ".join(ev["holdout"])),
]
protocol = pd.DataFrame(rows, columns=["Commitment", "Value"])
protocol.to_csv(TAB / "protocol.csv", index=False)
protocol

,Commitment,Value
0,Window,expanding from 2007-01-01
1,Step size,1 calendar year
2,Retraining cadence,annual (start of each validation/holdout year)
3,Label horizon,21 sessions (open t+1 -> open t+22)
4,Label buffer (purge),22 sessions
5,Feature buffer (embargo),0 (training never follows validation)
6,Longest feature lookback,252 sessions
7,Discovery (feature triage),2008-01-01 to 2014-12-31
8,HP tuning (fixed split),"train <= 2013-12-31, valid 2014-01-01 to 2014-..."
9,Validation folds,2015-2023 (one calendar year each)


In [3]:
folds = []
for y in ev["validation_folds"]:
    folds.append({"fold": y, "train_start": ev["first_train_date"], "train_end": f"{y}-01-01 minus {ev['label_buffer_sessions']} sessions",
                  "valid_start": f"{y}-01-01", "valid_end": f"{y}-12-31"})
for y in (2024, 2025):
    folds.append({"fold": f"holdout_{y}", "train_start": ev["first_train_date"], "train_end": f"{y}-01-01 minus {ev['label_buffer_sessions']} sessions",
                  "valid_start": f"{y}-01-01", "valid_end": f"{y}-12-31"})
pd.DataFrame(folds)

,fold,train_start,train_end,valid_start,valid_end
0,2015,2007-01-01,2015-01-01 minus 22 sessions,2015-01-01,2015-12-31
1,2016,2007-01-01,2016-01-01 minus 22 sessions,2016-01-01,2016-12-31
2,2017,2007-01-01,2017-01-01 minus 22 sessions,2017-01-01,2017-12-31
3,2018,2007-01-01,2018-01-01 minus 22 sessions,2018-01-01,2018-12-31
4,2019,2007-01-01,2019-01-01 minus 22 sessions,2019-01-01,2019-12-31
5,2020,2007-01-01,2020-01-01 minus 22 sessions,2020-01-01,2020-12-31
6,2021,2007-01-01,2021-01-01 minus 22 sessions,2021-01-01,2021-12-31
7,2022,2007-01-01,2022-01-01 minus 22 sessions,2022-01-01,2022-12-31
8,2023,2007-01-01,2023-01-01 minus 22 sessions,2023-01-01,2023-12-31
9,holdout_2024,2007-01-01,2024-01-01 minus 22 sessions,2024-01-01,2024-12-31


In [4]:
init_ledger()
print(LEDGER.read_text())

run_id,logged_at,strategy,setup_version,trial_family,trial,run,git_commit,data_snapshot,config,eval_window,metrics,decision,gate

